In [1]:
import pandas as pd
import numpy as np
import random
import warnings
from deap import base, creator, tools, algorithms
from sklearn.svm import SVR
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

In [ ]:
# load imputed data 


In [ ]:
# For this execution, we will focus on the Macro timeline (1990-2024) 
# to ensure a robust sample size and drop the highly sparse plant data.
df_macro = df[df['Year'] >= 1990].reset_index(drop=True)
plant_cols = [c for c in df_macro.columns if 'plant' in c]
df_macro = df_macro.drop(columns=plant_cols)
df_macro = df_macro.ffill().bfill() # Catch any minor remaining NaNs

# Define Target (y) and Features (X)
y = df_macro['pollinators_mean_occupancy'].values
drop_cols = ['Year', 'pollinators_mean_occupancy', 'pollinating_wild_bees_mean_occupancy', 'pollinating_hoverflies_mean_occupancy']
X_df = df_macro.drop(columns=[c for c in drop_cols if c in df_macro.columns])
X = X_df.values
feature_names = X_df.columns.tolist()
num_features = len(feature_names)


# =====================================================================
# 2. BASELINE MODEL (Without Feature Selection)
# =====================================================================
print("\n--- PHASE 2: Baseline Model (All Features) ---")

# Create the SVR Pipeline (StandardScaler + RBF Kernel, C=1.0, epsilon=0.1)
baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
])

# 5-Fold Cross Validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
baseline_scores = cross_val_score(baseline_model, X, y, cv=cv, scoring='neg_mean_squared_error')
baseline_rmse = np.sqrt(-baseline_scores.mean())

print(f"Total Features Used: {num_features}")
print(f"Baseline Cross-Validated RMSE: {baseline_rmse:.4f}")


# =====================================================================
# 3. OPTIMIZED MODEL (Genetic Algorithm Feature Selection)
# =====================================================================
print("\n--- PHASE 3: AI Optimization (Genetic Algorithm) ---")

# Define DEAP Framework (Minimizing RMSE)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, num_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_features(individual):
    selected_indices = [i for i in range(num_features) if individual[i] == 1]
    
    # Constraint: Must select at least 3 features to prevent spurious single-feature correlation
    if len(selected_indices) < 3:
        return (9999.0,)
        
    X_subset = X[:, selected_indices]
    
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))
    ])
    
    scores = cross_val_score(model, X_subset, y, cv=cv, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-scores.mean())
    
    # Fitness = RMSE + slight dimensionality penalty (0.02 per feature)
    fitness_score = rmse + (0.02 * len(selected_indices))
    return (fitness_score,)

toolbox.register("evaluate", evaluate_features)
# Parameters as defined in Section 3.6
toolbox.register("mate", tools.cxUniform, indpb=0.5)  # 50% Uniform Crossover
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# Initialize Population (50) and Run for 50 Generations
pop = toolbox.population(n=50)
hof = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("min", np.min)

print("Evolving Generations (Pop=50, Gen=50)...")
pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=50, 
                               stats=stats, halloffame=hof, verbose=False)

# Extract Best Individual
best_ind = hof[0]
optimized_rmse_with_penalty = best_ind.fitness.values[0]

# Calculate actual RMSE without the penalty for fair comparison
selected_indices = [i for i in range(num_features) if best_ind[i] == 1]
optimized_scores = cross_val_score(baseline_model, X[:, selected_indices], y, cv=cv, scoring='neg_mean_squared_error')
optimized_true_rmse = np.sqrt(-optimized_scores.mean())

selected_features = [feature_names[i] for i in selected_indices]

# =====================================================================
# 4. FINAL RESULTS OUTPUT
# =====================================================================
print("\n" + "="*50)
print("FINAL EVALUATION & COMPARISON")
print("="*50)
print(f"1. Baseline RMSE (Without Selection):   {baseline_rmse:.4f} ({num_features} features)")
print(f"2. Optimized RMSE (With GA Selection):  {optimized_true_rmse:.4f} ({len(selected_features)} features)")
print(f"Performance Improvement:                {baseline_rmse - optimized_true_rmse:.4f} RMSE points")
print("\nFeatures retained by the Genetic Algorithm:")
for feat in selected_features:
    print(f"  [+] {feat}")